In [2]:
# 🎯 Example 1: Sequential Multi-Agent System (Pipeline)
# 🧠 Scenario

# “A travel company has different employees (agents):

# Planner → decides steps

# Flight Agent → finds flights

# Weather Agent → checks weather

# Decision Agent → gives final answer”

# ================================
# AGENT 1: PLANNER
# ================================
def planner_agent(user_query):
    print("\n[Planner Agent] Creating plan...")
    return ["flight", "weather", "decision"]


# ================================
# AGENT 2: FLIGHT AGENT
# ================================
def flight_agent():
    print("\n[Flight Agent] Fetching flights...")
    return [
        {"airline": "IndiGo", "price": 4500},
        {"airline": "Air India", "price": 5200}
    ]


# ================================
# AGENT 3: WEATHER AGENT
# ================================
def weather_agent():
    print("\n[Weather Agent] Checking weather...")
    return {"condition": "Clear", "temp": 28}


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(flights, weather):
    print("\n[Decision Agent] Making decision...")

    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Book {cheapest['airline']} at ₹{cheapest['price']}"


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def travel_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    flights = None
    weather = None

    for step in plan:
        if step == "flight":
            flights = flight_agent()

        elif step == "weather":
            weather = weather_agent()

        elif step == "decision":
            result = decision_agent(flights, weather)

    return result


# RUN
response = travel_multi_agent("Plan my trip Delhi to Mumbai")
print("\nFinal Answer:", response) 

User Query: Plan my trip Delhi to Mumbai

[Planner Agent] Creating plan...

[Flight Agent] Fetching flights...

[Weather Agent] Checking weather...

[Decision Agent] Making decision...

Final Answer: Book IndiGo at ₹4500


In [24]:
!pip install groq

Defaulting to user installation because normal site-packages is not writeable


ModuleNotFoundError: No module named 'openai'

In [11]:
!pip install openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [23]:
import sys
import os
import json
from pathlib import Path
from typing import Dict, List, Any
from groq import Groq

BASE_DIR = Path().resolve()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

print("Current working directory:", BASE_DIR)
print("GROQ_API_KEY loaded:", bool(GROQ_API_KEY))
print("GROQ_MODEL loaded:", GROQ_MODEL)

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)

def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        max_completion_tokens=1024
    )

    return completion.choices[0].message.content.strip()

def safe_json_parse(text: str) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start != -1 and end != -1:
            return json.loads(text[start:end])
        raise ValueError("Model did not return valid JSON")


class IntakeAgent:
    def run(self, patient_data: Dict[str, Any]) -> Dict[str, Any]:
        system_prompt = (
            "You are an Intake Agent in a hospital workflow simulation. "
            "Review patient symptoms, history, vitals, and preferences. "
            "Decide the next steps needed. "
            "Return only valid JSON with keys: "
            "summary, required_steps, priority_level, intake_notes."
        )

        user_prompt = f"""
        Patient Data:
        {json.dumps(patient_data, indent=2)}

        Required output example:
        {{
          "summary": "...",
          "required_steps": ["blood_test", "xray", "consultation"],
          "priority_level": "medium",
          "intake_notes": "..."
        }}
        """

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DiagnosticAgent:
    def simulate_tests(self, patient_data: Dict[str, Any], plan: Dict[str, Any]) -> Dict[str, Any]:
        symptoms = [s.lower() for s in patient_data.get("symptoms", [])]
        vitals = patient_data.get("vitals", {})
        temperature = vitals.get("temperature_c", 98.6)
        spo2 = vitals.get("spo2", 99)

        lab_results = {
            "cbc": {
                "wbc": "high" if "fever" in symptoms or temperature > 99.5 else "normal",
                "hemoglobin": "normal"
            },
            "crp": "elevated" if "fever" in symptoms else "normal",
            "blood_sugar": "normal"
        }

        scan_results = {
            "chest_xray": "mild infiltrates" if "cough" in symptoms and spo2 < 97 else "clear",
            "ecg": "normal"
        }

        return {
            "ordered_tests": plan.get("required_steps", []),
            "lab_results": lab_results,
            "scan_results": scan_results
        }

    def run(self, patient_data: Dict[str, Any], intake_output: Dict[str, Any]) -> Dict[str, Any]:
        test_data = self.simulate_tests(patient_data, intake_output)

        system_prompt = (
            "You are a Diagnostic Agent in a hospital workflow simulation. "
            "Interpret patient symptoms, history, vitals, and simulated test results. "
            "Return only valid JSON with keys: "
            "possible_conditions, diagnostic_reasoning, confidence_level, red_flags."
        )

        user_prompt = f"""
        Patient Data:
        {json.dumps(patient_data, indent=2)}

        Intake Output:
        {json.dumps(intake_output, indent=2)}

        Test Data:
        {json.dumps(test_data, indent=2)}

        Required output example:
        {{
          "possible_conditions": ["Condition A", "Condition B"],
          "diagnostic_reasoning": "...",
          "confidence_level": "medium",
          "red_flags": ["..."]
        }}
        """

        result = call_llm(system_prompt, user_prompt)
        diagnosis = safe_json_parse(result)
        diagnosis["test_data"] = test_data
        return diagnosis


class TreatmentAgent:
    def run(self, patient_data: Dict[str, Any], diagnosis_output: Dict[str, Any]) -> Dict[str, Any]:
        system_prompt = (
            "You are a Treatment Agent in a hospital workflow simulation. "
            "Suggest treatment options based on likely diagnosis, symptoms, patient preference, and safety. "
            "Do not give extreme or unsafe recommendations. "
            "Return only valid JSON with keys: "
            "treatment_options, preferred_option, treatment_rationale, follow_up_advice."
        )

        user_prompt = f"""
        Patient Data:
        {json.dumps(patient_data, indent=2)}

        Diagnosis Output:
        {json.dumps(diagnosis_output, indent=2)}

        Required output example:
        {{
          "treatment_options": [
            "Option 1",
            "Option 2"
          ],
          "preferred_option": "Option 1",
          "treatment_rationale": "...",
          "follow_up_advice": "..."
        }}
        """

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DecisionAgent:
    def run(
        self,
        patient_data: Dict[str, Any],
        intake_output: Dict[str, Any],
        diagnosis_output: Dict[str, Any],
        treatment_output: Dict[str, Any]
    ) -> Dict[str, Any]:
        system_prompt = (
            "You are the final Decision Agent in a hospital workflow simulation. "
            "Review intake, diagnosis, and treatment options and produce a final recommendation. "
            "Return only valid JSON with keys: "
            "final_recommendation, patient_friendly_explanation, escalation_needed, final_notes."
        )

        user_prompt = f"""
        Patient Data:
        {json.dumps(patient_data, indent=2)}

        Intake Output:
        {json.dumps(intake_output, indent=2)}

        Diagnosis Output:
        {json.dumps(diagnosis_output, indent=2)}

        Treatment Output:
        {json.dumps(treatment_output, indent=2)}

        Required output example:
        {{
          "final_recommendation": "...",
          "patient_friendly_explanation": "...",
          "escalation_needed": false,
          "final_notes": "..."
        }}
        """

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HospitalMultiAgentSystem:
    def __init__(self):
        self.intake_agent = IntakeAgent()
        self.diagnostic_agent = DiagnosticAgent()
        self.treatment_agent = TreatmentAgent()
        self.decision_agent = DecisionAgent()

    def run(self, patient_data: Dict[str, Any]) -> Dict[str, Any]:
        intake_output = self.intake_agent.run(patient_data)
        diagnosis_output = self.diagnostic_agent.run(patient_data, intake_output)
        treatment_output = self.treatment_agent.run(patient_data, diagnosis_output)
        decision_output = self.decision_agent.run(
            patient_data,
            intake_output,
            diagnosis_output,
            treatment_output
        )

        return {
            "patient_data": patient_data,
            "intake_output": intake_output,
            "diagnosis_output": diagnosis_output,
            "treatment_output": treatment_output,
            "decision_output": decision_output
        }


if __name__ == "__main__":
    patient_case = {
        "name": "Rahul Sharma",
        "age": 45,
        "gender": "Male",
        "symptoms": ["fever", "cough", "fatigue"],
        "medical_history": ["hypertension"],
        "allergies": ["penicillin"],
        "vitals": {
            "temperature_c": 101.2,
            "blood_pressure": "140/90",
            "heart_rate": 96,
            "spo2": 95
        },
        "preferences": {
            "prefers_non_surgical": True,
            "prefers_home_recovery_if_possible": True
        }
    }

    system = HospitalMultiAgentSystem()
    output = system.run(patient_case)

    print("\n" + "=" * 70)
    print("HOSPITAL SEQUENTIAL MULTI-AGENT SYSTEM OUTPUT")
    print("=" * 70)

    print("\n1. Intake Agent Output")
    print(json.dumps(output["intake_output"], indent=2))

    print("\n2. Diagnostic Agent Output")
    print(json.dumps(output["diagnosis_output"], indent=2))

    print("\n3. Treatment Agent Output")
    print(json.dumps(output["treatment_output"], indent=2))

    print("\n4. Decision Agent Output")
    print(json.dumps(output["decision_output"], indent=2))

In [ ]:
# Prime number program
from math import isqrt

def is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False
    limit = isqrt(n)
    for d in range(3, limit + 1, 2):
        if n % d == 0:
            return False
    return True

# Input + output
try:
    n = int(input("Enter a number: "))
    print("Prime" if is_prime(n) else "Not Prime")
except ValueError:
    print("Invalid input. Please enter an integer.")
